In [1]:
import pandas as pd
import glob
import os
import numpy as np
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import IterativeImputer

# Read CSV File Data Curah Hujan

In [2]:
folder_curah_hujan = r"C:\Users\Giana\OneDrive\Dokumen\!!! Tugas Akhir\Data\Curah Hujan\DATA INI YANG DIPAKE"
file_curah_hujan = glob.glob(os.path.join(folder_curah_hujan, "*.csv"))

### Pengecekan Missing Value pada Tanggal

In [3]:
# Map untuk menyimpan hasil pengecekan data curah hujan
laporan_pengecekan = {}

# Pengecekan tanggal pada masing-masing file
for file_path_curah_hujan in file_curah_hujan:
    file_name_curah_hujan = os.path.basename(file_path_curah_hujan)
    data_curah_hujan = pd.read_csv(file_path_curah_hujan)
    data_curah_hujan.columns = data_curah_hujan.columns.str.strip().str.upper()
    
    data_curah_hujan["DATE"] = pd.to_datetime(data_curah_hujan["DATE"])
    tanggal_awal = data_curah_hujan["DATE"].min()
    tanggal_akhir = data_curah_hujan["DATE"].max()
    kalender_ideal = pd.date_range(start=tanggal_awal, end=tanggal_akhir)
    tanggal_hilang = kalender_ideal.difference(data_curah_hujan["DATE"])
    
    laporan_pengecekan[file_name_curah_hujan] = {
        "Total_Baris": len(data_curah_hujan),
        "Harusnya": len(kalender_ideal),
        "Tanggal_Hilang": tanggal_hilang.strftime('%Y-%m-%d').tolist()
    }

    if not tanggal_hilang.empty:
        print(f"File: {os.path.basename(file_path_curah_hujan)}")
        print(f"❌ Ditemukan {len(tanggal_hilang)} hari yang hilang:")
        # Menampilkan daftar tanggalnya
        for tgl in tanggal_hilang:
            print(f"   - {tgl.strftime('%Y-%m-%d')}")
    else:
        print(f"File: {os.path.basename(file_path_curah_hujan)} ✅ LENGKAP")

File: Curah Hujan 2015.csv
❌ Ditemukan 1 hari yang hilang:
   - 2015-03-20
File: Curah Hujan 2016.csv ✅ LENGKAP
File: Curah Hujan 2017.csv
❌ Ditemukan 1 hari yang hilang:
   - 2017-11-22
File: Curah Hujan 2018.csv
❌ Ditemukan 1 hari yang hilang:
   - 2018-09-17
File: Curah Hujan 2019.csv ✅ LENGKAP
File: Curah Hujan 2020.csv ✅ LENGKAP
File: Curah Hujan 2021.csv
❌ Ditemukan 4 hari yang hilang:
   - 2021-02-11
   - 2021-02-12
   - 2021-02-20
   - 2021-02-21
File: Curah Hujan 2022.csv
❌ Ditemukan 2 hari yang hilang:
   - 2022-05-30
   - 2022-10-24
File: Curah Hujan 2023.csv
❌ Ditemukan 3 hari yang hilang:
   - 2023-02-02
   - 2023-08-12
   - 2023-09-03
File: Curah Hujan 2024.csv
❌ Ditemukan 13 hari yang hilang:
   - 2024-03-28
   - 2024-03-29
   - 2024-03-30
   - 2024-03-31
   - 2024-04-01
   - 2024-04-02
   - 2024-05-01
   - 2024-08-12
   - 2024-11-18
   - 2024-11-19
   - 2024-11-20
   - 2024-11-21
   - 2024-12-01
File: Curah Hujan Agustus 2024.csv ✅ LENGKAP
File: Curah Hujan April 2024.c

C:\Users\Giana\AppData\Local\Temp\ipykernel_16436\2939988137.py:10: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data_curah_hujan["DATE"] = pd.to_datetime(data_curah_hujan["DATE"])
C:\Users\Giana\AppData\Local\Temp\ipykernel_16436\2939988137.py:10: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data_curah_hujan["DATE"] = pd.to_datetime(data_curah_hujan["DATE"])


### Mengubah satuan di data NOAA sesuai Standar Indonesia

In [4]:
# Mendefiniskan singkatan nama pada data agar mudah dipahami
nama_singkatan_noaa = {
    "PRCP" : "curah_hujan",
    "DEWP" : "dew_point",
    "WDSP" : "kecepatan_angin",
    "TEMP" : "suhu",
    "DATE" : "tanggal"
}

# Membuat list nama bukan untuk membedakan file data NOAA dan BMKG
list_bulan = [
    "januari", "februari", "maret", "april", "mei", "juni", "juli", "agustus", "september", "oktober", "november", "desember"
]
list_data_noaa = []

for file_path in file_curah_hujan:
    file_nama = os.path.basename(file_path).lower()
    
    file_bmkg = any(bulan in file_nama for bulan in list_bulan)
    if not file_bmkg:
        file_noaa = pd.read_csv(file_path)
        file_noaa.columns = file_noaa.columns.str.strip().str.upper()
        file_noaa = file_noaa.rename(columns=nama_singkatan_noaa)
        
        # Tanggal
        file_noaa["tanggal"] = pd.to_datetime(file_noaa["tanggal"])
        
        # Curah hujan // mm
        if "curah_hujan" in file_noaa.columns:
            file_noaa["curah_hujan"] = file_noaa["curah_hujan"].replace([99.99, 9999], np.nan)
            file_noaa["curah_hujan"] = (file_noaa["curah_hujan"] * 25.4).round(1)
            
        # Dew Point // %
        if "dew_point" in file_noaa.columns:
            file_noaa["dew_point"] = (file_noaa["dew_point"] - 32) * 5/9
        
        # Suhu (sekaligus untuk menghitung kelembapan udara di data NOAA) // C
        if "suhu" in file_noaa.columns:
            file_noaa["suhu"] = file_noaa["suhu"].replace([9999.9, 9999], np.nan)
            file_noaa["suhu"] = ((file_noaa["suhu"] - 32) * 5/9).round(1)
        
        if "suhu" in file_noaa.columns and "dew_point" in file_noaa.columns:
            file_noaa["kelembapan_udara"] = (100 * (
                np.exp((17.625 * file_noaa["dew_point"]) / (243.04 + file_noaa["dew_point"])) /
                np.exp((17.625 * file_noaa["suhu"]) / (243.04 + file_noaa["suhu"]))
            )).round(1)
            
        # Kecepatan angin // m/s
        if "kecepatan_angin" in file_noaa.columns:
            file_noaa["kecepatan_angin"] = (
                file_noaa["kecepatan_angin"] * 0.514444
            ).round(0)
            
        kolom_final = ["tanggal", "curah_hujan", "suhu", "kelembapan_udara", "kecepatan_angin"]
                
        file_noaa = file_noaa[kolom_final]
        list_data_noaa.append(file_noaa)
        
        print(file_noaa[["tanggal", "curah_hujan", "suhu", "kelembapan_udara", "kecepatan_angin"]].head())
        
# Menggabungkan file data NOAA
if list_data_noaa:
    data_noaa_final = pd.concat(list_data_noaa, ignore_index=True)
    print(f"Total Data BMKG: {len(data_noaa_final)} baris")
    print(data_noaa_final.head())

     tanggal  curah_hujan  suhu  kelembapan_udara  kecepatan_angin
0 2015-01-01          7.1  26.4              81.6              2.0
1 2015-01-02         30.0  26.4              82.7              2.0
2 2015-01-03          8.9  26.0              84.1              3.0
3 2015-01-04          2.0  25.8              83.4              2.0
4 2015-01-05          0.0  28.0              72.1              3.0
     tanggal  curah_hujan  suhu  kelembapan_udara  kecepatan_angin
0 2016-01-01          0.0  28.1              77.7              2.0
1 2016-01-02         15.2  27.4              83.4              2.0
2 2016-01-03          0.0  29.0              75.7              2.0
3 2016-01-04          0.0  29.2              78.4              2.0
4 2016-01-05         11.9  28.4              78.1              1.0
     tanggal  curah_hujan  suhu  kelembapan_udara  kecepatan_angin
0 2017-01-01          0.0  30.1              65.8              3.0
1 2017-01-02          0.0  30.8              62.8             

### Penggabungan Data 2024 pada NOAA dan BMKG

In [5]:
list_data_bmkg = []
kolom_final = ["tanggal", "curah_hujan", "suhu", "kelembapan_udara", "kecepatan_angin"]

for file_path in file_curah_hujan:
    file_nama = os.path.basename(file_path).lower()
    print("Cek file:", file_nama)

    if any(bulan in file_nama for bulan in list_bulan):

        data_bmkg = pd.read_csv(file_path)
        data_bmkg = data_bmkg.rename(columns=nama_singkatan_noaa)

        data_bmkg.columns = (data_bmkg.columns.str.strip().str.lower())

        print("Kolom bersih:", data_bmkg.columns.tolist())        

        if "tanggal" not in data_bmkg.columns or "curah_hujan" not in data_bmkg.columns:
            print("❌ Struktur kolom tidak sesuai")
            continue

        data_bmkg["tanggal"] = pd.to_datetime(
            data_bmkg["tanggal"], errors="coerce", dayfirst=True
        )
        
        mask = data_bmkg["tanggal"].isna()
        data_bmkg.loc[mask, "tanggal"] = pd.to_datetime(
            data_bmkg.loc[mask, "tanggal"],
            errors="coerce",
            dayfirst=False
        )

        data_bmkg = data_bmkg[data_bmkg["tanggal"].dt.year == 2024] 

        kolom_tersedia = [kolom for kolom in kolom_final if kolom in data_bmkg.columns]
        list_data_bmkg.append(data_bmkg[kolom_final])

Cek file: curah hujan 2015.csv
Cek file: curah hujan 2016.csv
Cek file: curah hujan 2017.csv
Cek file: curah hujan 2018.csv
Cek file: curah hujan 2019.csv
Cek file: curah hujan 2020.csv
Cek file: curah hujan 2021.csv
Cek file: curah hujan 2022.csv
Cek file: curah hujan 2023.csv
Cek file: curah hujan 2024.csv
Cek file: curah hujan agustus 2024.csv
Kolom bersih: ['id wmo', 'lintang', 'bujur', 'elevasi', 'tanggal', 'suhu', 'kelembapan_udara', 'curah_hujan', 'kecepatan_angin']
Cek file: curah hujan april 2024.csv
Kolom bersih: ['id wmo', 'lintang', 'bujur', 'elevasi', 'tanggal', 'suhu', 'kelembapan_udara', 'curah_hujan', 'kecepatan_angin']
Cek file: curah hujan desember 2024.csv
Kolom bersih: ['id wmo', 'lintang', 'bujur', 'elevasi', 'tanggal', 'suhu', 'kelembapan_udara', 'curah_hujan', 'kecepatan_angin']
Cek file: curah hujan maret 2024.csv
Kolom bersih: ['id wmo', 'lintang', 'bujur', 'elevasi', 'tanggal', 'suhu', 'kelembapan_udara', 'curah_hujan', 'kecepatan_angin']
Cek file: curah hujan

In [6]:
if list_data_bmkg:
    data_bmkg_final = pd.concat(list_data_bmkg, ignore_index=True)
    print("Total data BMKG:", len(data_bmkg_final))
    display(data_bmkg_final.head())

Total data BMKG: 13


,tanggal,curah_hujan,suhu,kelembapan_udara,kecepatan_angin
0,2024-08-12,0.0,29.0,62,1
1,2024-01-04,18.2,29.2,78,1
2,2024-02-04,1.2,29.5,77,1
3,2024-12-01,0.6,30.4,68,3
4,2024-03-28,NaN,29.3,77,1


In [7]:
# Mengabungkan final data noaa dan bmkg
if "data_noaa_final" and "data_bmkg_final" in globals():
    data_curah_hujan_final = (pd.concat([data_noaa_final, data_bmkg_final], ignore_index=True)).sort_values("tanggal").drop_duplicates("tanggal").reset_index(drop=True)
    
    display(data_curah_hujan_final.head())

,tanggal,curah_hujan,suhu,kelembapan_udara,kecepatan_angin
0,2015-01-01,7.1,26.4,81.6,2.0
1,2015-01-02,30.0,26.4,82.7,2.0
2,2015-01-03,8.9,26.0,84.1,3.0
3,2015-01-04,2.0,25.8,83.4,2.0
4,2015-01-05,0.0,28.0,72.1,3.0


### Pengisian Missing Value

In [8]:
output_file = "Data Gabung Curah Hujan.csv"
    
# Ubah nilai 99.99 agar dikategorikan NA
data_curah_hujan_final = data_curah_hujan_final.replace([99.99, 9999], np.nan)

# Menghitung jumlah missing value
total_na = data_curah_hujan_final.isna().sum()
print(f"Jumlah missing value pada masing-masing index adalah: \n{total_na}")

Jumlah missing value pada masing-masing index adalah: 
tanggal               0
curah_hujan         212
suhu                  0
kelembapan_udara      0
kecepatan_angin       0
dtype: int64


In [9]:
# IterativeImputer
# Menentukan kolom numerik
imputer = IterativeImputer(max_iter=10, random_state=42)
imputed_data = imputer.fit_transform(data_curah_hujan_final[["curah_hujan", "kelembapan_udara", "kecepatan_angin", "suhu"]])
data_curah_hujan_imputed = pd.DataFrame(imputed_data, columns=("curah_hujan", "kelembapan_udara", "kecepatan_angin", "suhu"))
data_curah_hujan_imputed["tanggal"] = data_curah_hujan_final["tanggal"]

# Mengnolkan nilai negatif
data_curah_hujan_imputed[["curah_hujan", "kelembapan_udara", "kecepatan_angin", "suhu"]] = data_curah_hujan_imputed[["curah_hujan", "kelembapan_udara", "kecepatan_angin", "suhu"]].clip(lower=0)
data_curah_hujan_imputed = data_curah_hujan_imputed[["tanggal"] + ["curah_hujan", "kelembapan_udara", "kecepatan_angin", "suhu"]]

data_curah_hujan_imputed.isna().sum()

tanggal             0
curah_hujan         0
kelembapan_udara    0
kecepatan_angin     0
suhu                0
dtype: int64

### Ubah data harian ke data bulanan

In [10]:
data_curah_hujan_imputed["tanggal"] = pd.to_datetime(data_curah_hujan_imputed["tanggal"])
data_curah_hujan_imputed.set_index("tanggal", inplace=True)

data_curah_hujan_bulanan = data_curah_hujan_imputed.resample("MS").agg({
    "curah_hujan": "sum",
    "kelembapan_udara": "mean",
    "kecepatan_angin": "mean",
    "suhu": "mean"
})

display(data_curah_hujan_bulanan.head())

,curah_hujan,kelembapan_udara,kecepatan_angin,suhu
tanggal,,,,
2015-01-01,219.088643,80.200000,1.548387,27.058065
2015-02-01,588.560036,83.092857,1.142857,26.814286
2015-03-01,206.633419,77.826667,1.100000,27.853333
2015-04-01,80.568713,75.606667,1.200000,28.640000
2015-05-01,21.113034,71.141935,1.032258,29.370968


### Splitting Data

In [11]:
file_gabung  = data_curah_hujan_bulanan.reset_index()
file_gabung.to_csv("Data Gabung Curah Hujan Bulanan.csv", index=False)
file_gabung.head()

,tanggal,curah_hujan,kelembapan_udara,kecepatan_angin,suhu
0,2015-01-01,219.088643,80.200000,1.548387,27.058065
1,2015-02-01,588.560036,83.092857,1.142857,26.814286
2,2015-03-01,206.633419,77.826667,1.100000,27.853333
3,2015-04-01,80.568713,75.606667,1.200000,28.640000
4,2015-05-01,21.113034,71.141935,1.032258,29.370968


In [12]:
training = file_gabung[:int(0.6 * len (file_gabung))]
validasi = file_gabung[int(0.6 * len (file_gabung)):int(0.8 * len (file_gabung))]
testing = file_gabung[int(0.8 * len (file_gabung)):]

print(f"Banyak data training adalah {len(training)}")
print(f"Banyak data validasi adalah {len(validasi)}")
print(f"Banyak data testing adalah {len(testing)}")

Banyak data training adalah 72
Banyak data validasi adalah 24
Banyak data testing adalah 24


In [13]:
# Simpan split data
training.to_csv("Dataset_Train_Bulanan.csv", index=False)
validasi.to_csv("Dataset_Validation_Bulanan.csv", index=False)
testing.to_csv("Dataset_Testing_Bulanan.csv", index=False)